# Custom Transformers & Feature Engineering — Giải thích chi tiết



 Cấu trúc code hiện tại:

1. Kiến trúc tổng thể và vai trò của `custom_transformers.py`
2. Section 1 — Haversine + `pandas_udf` + Arrow + DOT/IATA fallback
3. Section 2 — `OutlierIQRTruncator` theo mô hình **Estimator → Model**
4. Section 3 — Leakage-safe `TargetEncoder`
5. Section 4 — Scheduled hour + route congestion
6. Section 5 — 5 cấp độ feature: Airline / Airport / Airplane / Flight / Temporal
7. Leakage: cái gì được phép dùng và cái gì phải cắt theo thời điểm
8. Quan hệ giữa `custom_transformers.py` và `build_modeling_dataset.py`
9. Checklist để kiểm tra trước khi dùng dataset cho model

**Quan trọng:** Notebook ưu tiên các dataset toy nhỏ để có thể chạy nhanh và nhìn thấy kết quả. Phần full dataset hàng triệu dòng chỉ được gọi ở các cell tùy chọn.

## 0. Bức tranh lớn

`custom_transformers.py` là **nơi chứa logic feature engineering**. File `build_modeling_dataset.py` chỉ đóng vai trò orchestration: đọc `base`, gọi các hàm trong `custom_transformers.py`, merge feature vào `base`, checkpoint Parquet rồi chuyển sang nhóm feature tiếp theo.

Grain cuối cùng của `base` là:

> **1 dòng = 1 chuyến bay**

Composite key dùng để merge feature:

`AIRLINE + FLIGHT_NUMBER + YEAR + MONTH + DAY + SCHEDULED_DEPARTURE`

Điểm quan trọng nhất về kiến trúc:

```text
RAW / typed Parquet
        │
        ▼
      BASE
  1 row / flight
        │
        ├── Haversine
        ├── Hour + Congestion
        ├── Airline-level
        ├── Airport-level
        ├── Airplane-level
        ├── Flight-level
        └── Temporal-level
        │
        ▼
   MODELING DATASET
        │
        ├── train
        └── test
```

Cách orchestration này giúp cắt Spark lineage dài và giảm áp lực memory trên máy local.

## 1. Chuẩn bị môi trường

Notebook giả định:

- Notebook nằm cùng project với `custom_transformers.py`.
- Python environment đã có PySpark, pandas, NumPy và PyArrow.
- Nếu notebook được mở từ thư mục khác, có thể thêm project root vào `sys.path`.

**Không cần chạy full dataset để hiểu code.** Các ví dụ phía dưới chủ yếu dùng DataFrame rất nhỏ.

In [1]:
import os
import sys
from pathlib import Path

# Nếu notebook nằm trong project root thì dòng này không cần thay đổi.
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)


Python: c:\Users\dohuu\miniconda3\envs\spark_env\python.exe
Project root: f:\DSEB\Semester7\Big Data\BigData_Group1\notebooks


In [2]:
from pyspark.sql import SparkSession, functions as F, Window
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("CustomTransformers-Demo")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "true")
    .config("spark.python.worker.connect.timeout", "60")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Python:", sys.executable)
print("Spark:", spark.version)


Python: c:\Users\dohuu\miniconda3\envs\spark_env\python.exe
Spark: 3.5.9


# 2. SECTION 1 — Haversine Distance

## 2.1 Bài toán

Mỗi chuyến bay có:

- sân bay đi `ORIGIN_AIRPORT`
- sân bay đến `DESTINATION_AIRPORT`

Bảng `airports` cung cấp:

- `IATA_CODE`
- `LATITUDE`
- `LONGITUDE`

Ta muốn tạo khoảng cách đường chim bay giữa hai sân bay.

Công thức Haversine:

$$
a = \sin^2\left(\frac{\Delta\varphi}{2}\right)
+ \cos(\varphi_1)\cos(\varphi_2)
\sin^2\left(\frac{\Delta\lambda}{2}\right)
$$

$$
c = 2\arcsin(\sqrt{a})
$$

$$
d = R c
$$

Trong code:

- $R = 6371.0088$ km
- latitude/longitude phải được đổi sang radian
- kết quả là km

### Vì sao không loop từng dòng?

Dataset có hàng triệu chuyến bay. Python UDF chạy từng row có overhead lớn.

Code dùng:

```python
@F.pandas_udf(DoubleType())
```

Spark truyền dữ liệu theo **batch** sang Python worker thông qua Arrow. Sau đó NumPy xử lý vectorized trên cả `Series`.

In [3]:
import sys
sys.path.append("..")

In [4]:
from custom_transformers import haversine_km

print(haversine_km)


<function haversine_km at 0x000001A5C856DDA0>


In [5]:
# Ví dụ nhỏ: Hà Nội -> TP.HCM gần đúng về mặt minh họa.
# Tọa độ chỉ để demo công thức, không phải dữ liệu flight thật.
demo = spark.createDataFrame(
    [
        (21.0285, 105.8542, 10.8231, 106.6297),
        (40.6413, -73.7781, 33.9416, -118.4085),
        (None, 106.6297, 10.8231, 106.6297),
    ],
    ["lat1", "lon1", "lat2", "lon2"]
)

demo2 = demo.withColumn(
    "distance_km",
    haversine_km("lat1", "lon1", "lat2", "lon2")
)

demo2.show()


+-------+--------+-------+---------+------------------+
|   lat1|    lon1|   lat2|     lon2|       distance_km|
+-------+--------+-------+---------+------------------+
|21.0285|105.8542|10.8231| 106.6297|1137.8059473947567|
|40.6413|-73.7781|33.9416|-118.4085|3974.3416895777727|
|   NULL|106.6297|10.8231| 106.6297|              NULL|
+-------+--------+-------+---------+------------------+



### 2.2 Kiểm tra input không hợp lệ

`haversine_km()` không chỉ tính công thức. Nó còn kiểm tra domain:

- latitude phải nằm trong `[-90, 90]`
- longitude phải nằm trong `[-180, 180]`
- nếu tọa độ null hoặc invalid → trả về `NaN`

Đây là một ví dụ về **defensive feature engineering**: transformer không giả định dữ liệu luôn sạch.

In [6]:
invalid_demo = spark.createDataFrame(
    [
        (91.0, 0.0, 10.0, 10.0),
        (10.0, 181.0, 10.0, 10.0),
        (None, 0.0, 10.0, 10.0),
    ],
    ["lat1", "lon1", "lat2", "lon2"]
)

invalid_demo.withColumn(
    "distance_km",
    haversine_km("lat1", "lon1", "lat2", "lon2")
).show()


+----+-----+----+----+-----------+
|lat1| lon1|lat2|lon2|distance_km|
+----+-----+----+----+-----------+
|91.0|  0.0|10.0|10.0|       NULL|
|10.0|181.0|10.0|10.0|       NULL|
|NULL|  0.0|10.0|10.0|       NULL|
+----+-----+----+----+-----------+



## 2.3 Vì sao cần fallback?

Code cuối cùng không phụ thuộc hoàn toàn vào Haversine.

Một phần dữ liệu dùng mã sân bay DOT numeric thay vì IATA. Khi không map được sang IATA → không lấy được latitude/longitude → không tính Haversine.

Vì vậy code tạo:

- `haversine_distance_km`: khoảng cách Haversine nếu có tọa độ
- `route_distance_km`: `DISTANCE * 1.60934`
- `has_haversine_distance`: có tính được Haversine hay không
- `distance_feature_km`: feature cuối cùng dùng cho model

Logic:

```text
Có đủ tọa độ?
    │
   YES ──> haversine_distance_km
    │
   NO
    │
    └────> route_distance_km
```

Và:

```python
distance_feature_km =
    coalesce(haversine_distance_km, route_distance_km)
```

Điểm hay của thiết kế này là **không biến việc thiếu tọa độ thành mất luôn feature khoảng cách**.

## 2.4 Airport mapping

Trong phiên bản cuối, `build_airport_code_mapping()` có thứ tự ưu tiên:

1. Direct code đã là IATA
2. Level 1: state + city + airport name
3. Level 2: state + city
4. Level 3: state + airport name
5. Manual mapping cho các code còn sót lại

Mapping được dựng một lần trong orchestration rồi truyền vào `add_haversine_distance_feature()`.

Đây là cách tách trách nhiệm:

- `custom_transformers.py`: **logic mapping**
- `build_modeling_dataset.py`: **khi nào gọi mapping và truyền bảng nào vào**

# 3. SECTION 2 — OutlierIQRTruncator

## 3.1 IQR là gì?

Với một numeric feature:

- $Q1$ = percentile 25%
- $Q3$ = percentile 75%
- $IQR = Q3 - Q1$

Bound chuẩn:

$$
Lower = Q1 - 1.5 \times IQR
$$

$$
Upper = Q3 + 1.5 \times IQR
$$

Ví dụ:

```text
Q1 = 100
Q3 = 200
IQR = 100

Lower = 100 - 1.5*100 = -50
Upper = 200 + 1.5*100 = 350
```

Giá trị 500 nằm ngoài upper bound nên được xem là outlier theo rule này.

### Nhưng tại sao code không chỉ là Transformer?

Đây là điểm quan trọng nhất của Section 2.

`Q1`, `Q3` phải được **học từ train/fold**, không được tính từ test.

Do đó thiết kế là:

```text
OutlierIQRTruncator (Estimator)
          │ fit(train)
          ▼
OutlierIQRTruncatorModel (Model)
          │ transform(train/test)
          ▼
       dữ liệu
```

Nếu tính Q1/Q3 trên toàn bộ dataset trước khi split, test data đã ảnh hưởng preprocessing → leakage.

In [7]:
from custom_transformers import OutlierIQRTruncator

toy_outlier = spark.createDataFrame(
    [
        (1.0,), (2.0,), (2.0,), (3.0,), (3.0,),
        (4.0,), (4.0,), (5.0,), (100.0,)
    ],
    ["x"]
)

toy_outlier.show()


+-----+
|    x|
+-----+
|  1.0|
|  2.0|
|  2.0|
|  3.0|
|  3.0|
|  4.0|
|  4.0|
|  5.0|
|100.0|
+-----+



In [8]:
outlier_estimator = OutlierIQRTruncator(
    inputCols=["x"],
    outputCols=["x_clipped"],
    iqrMultiplier=1.5,
    outlierFlagCols=["x_is_outlier"]
)

outlier_model = outlier_estimator.fit(toy_outlier)

print("IQR bounds learned from fit data:")
outlier_model.bounds_summary()


IQR bounds learned from fit data:


,output_col,input_col,q1,q3,lower_bound,upper_bound,flag_col
0,x_clipped,x,2.0,4.0,-1.0,7.0,x_is_outlier


In [9]:
outlier_result = outlier_model.transform(toy_outlier)
outlier_result.orderBy("x").show()


+-----+---------+------------+
|    x|x_clipped|x_is_outlier|
+-----+---------+------------+
|  1.0|      1.0|           1|
|  2.0|      2.0|           0|
|  2.0|      2.0|           0|
|  3.0|      3.0|           0|
|  3.0|      3.0|           0|
|  4.0|      4.0|           0|
|  4.0|      4.0|           0|
|  5.0|      5.0|           1|
|100.0|      7.0|           1|
+-----+---------+------------+



### 3.2 Hai output có ý nghĩa khác nhau

Với mỗi input column, model có thể tạo:

1. `x_clipped`
   - giá trị đã bị cắt về `[lower_bound, upper_bound]`
2. `x_is_outlier`
   - `1` nếu **giá trị gốc** nằm ngoài `[Q1, Q3]`
   - `0` nếu không


Điều này giữ lại giá trị gốc của feature và chỉ thêm thông tin “quan sát này có bất thường hay không”.

## 3.3 `approxQuantile()` và distributed computation

Code dùng:

```python
dataset.approxQuantile(
    in_col,
    [0.25, 0.75],
    relativeError=0.01
)
```

Không collect toàn bộ hàng triệu giá trị về Driver.

Ý tưởng:

```text
Executor 1 ─┐
Executor 2 ─┼─> distributed approximate quantile
Executor 3 ─┤
Executor 4 ─┘
             │
             ▼
          Q1, Q3
```

Sau khi fit, chỉ cần lưu các con số nhỏ:

```text
input_col
q1
q3
lower
upper
flag_col
```

Code serialize chúng thành JSON trong `boundsMapJson` để Spark ML có thể persistence model.

# 4. SECTION 3 — Leakage-safe Target Encoding

## 4.1 Target Encoding là gì?

Với categorical feature có nhiều category, ví dụ:

```text
AIRLINE
AA
DL
UA
...
```

Ta có thể thay category bằng một thống kê của target.

Ví dụ regression:

$$
TE(category) = E[Y \mid category]
$$

Nếu `AA` có delay trung bình 12 phút, encoding của `AA` sẽ gần 12.

Điểm mạnh:

- biến categorical thành numeric
- giữ được thông tin liên quan tới target
- hữu ích cho high-cardinality categorical variables

Nhưng có một vấn đề rất lớn:

> Nếu tính mean target bằng cả train + test, target của test đã đi ngược vào feature → target leakage.

Vì vậy `TargetEncoder` là **Estimator**, không phải một hàm transform đơn giản.

In [10]:
from custom_transformers import TargetEncoder

te_train = spark.createDataFrame(
    [
        ("A", 10.0),
        ("A", 20.0),
        ("A", 30.0),
        ("B", 5.0),
        ("B", 15.0),
        ("C", 100.0),
    ],
    ["category", "target"]
)

te_train.show()


+--------+------+
|category|target|
+--------+------+
|       A|  10.0|
|       A|  20.0|
|       A|  30.0|
|       B|   5.0|
|       B|  15.0|
|       C| 100.0|
+--------+------+



## 4.2 Công thức smoothing

Code không dùng mean thô mà dùng m-estimate:

$$
TE_c =
\frac{
n_c \cdot \bar{y}_c + m \cdot \bar{y}_{global}
}{
n_c + m
}
$$

Trong đó:

- $n_c$: số quan sát của category
- $\bar{y}_c$: target mean của category
- $\bar{y}_{global}$: mean target toàn train
- $m$: smoothing, mặc định `10.0`

### Trực giác

Category có ít dữ liệu:

```text
n_c nhỏ
```

→ trọng số của global mean lớn.

Category có nhiều dữ liệu:

```text
n_c lớn
```

→ category mean có ảnh hưởng lớn hơn.

Điều này tránh việc một category chỉ xuất hiện 1–2 lần tạo ra encoding cực đoan.

In [11]:
te = TargetEncoder(
    inputCols=["category"],
    outputCols=["category_te"],
    targetCol="target",
    smoothing=10.0
)

te_model = te.fit(te_train)

print("Một số encoding đã học:")
te_model.encoding_summary("category_te", top_n=10)


Một số encoding đã học:


,category,encoded_value,global_mean
0,C,36.363636,30.0
1,A,27.692308,30.0
2,B,26.666667,30.0


In [12]:
encoded_train = te_model.transform(te_train)
encoded_train.show()


+--------+------+------------------+
|category|target|       category_te|
+--------+------+------------------+
|       A|  10.0|27.692307692307693|
|       A|  20.0|27.692307692307693|
|       A|  30.0|27.692307692307693|
|       B|   5.0|26.666666666666668|
|       B|  15.0|26.666666666666668|
|       C| 100.0| 36.36363636363637|
+--------+------+------------------+



## 4.3 Unseen category

Giả sử test có category `D` nhưng train chưa từng có `D`.

Không thể tính:

$$
E[Y \mid D]
$$

từ train.

Code xử lý:

```text
unseen category
      │
      ▼
global_mean
```

Do đó không tạo `NULL` và không crash.

Đây là hành vi rất quan trọng khi deploy hoặc khi inference gặp category mới.

In [13]:
te_test = spark.createDataFrame(
    [
        ("A", 999.0),  # target ở đây chỉ để minh họa; transform không dùng target
        ("D", 123.0),  # unseen category
    ],
    ["category", "target"]
)

te_model.transform(te_test).show()


+--------+------+------------------+
|category|target|       category_te|
+--------+------+------------------+
|       A| 999.0|27.692307692307693|
|       D| 123.0|              30.0|
+--------+------+------------------+



# 5. SECTION 4 — Feature cơ bản

## 5.1 `scheduled_hour`

Input:

```text
SCHEDULED_DEPARTURE = HHMM
```

Ví dụ:

```text
0830 -> 8
1545 -> 15
2400 -> 0
```

`2400` là convention đặc biệt của dataset. Hàm quy nó về giờ `0` cho mục đích feature hour.

In [14]:
from custom_transformers import add_scheduled_hour_feature

hour_demo = spark.createDataFrame(
    [("0830",), ("1545",), ("2400",), ("0055",)],
    ["SCHEDULED_DEPARTURE"]
)

add_scheduled_hour_feature(hour_demo).show()


+-------------------+--------------+
|SCHEDULED_DEPARTURE|scheduled_hour|
+-------------------+--------------+
|               0830|             8|
|               1545|            15|
|               2400|             0|
|               0055|             0|
+-------------------+--------------+



## 5.2 `route_congestion_count`

Feature này đếm số chuyến được lên lịch tại cùng:

- `ORIGIN_AIRPORT`
- `YEAR`
- `MONTH`
- `DAY`
- `scheduled_hour`

Nó là **schedule-based feature**, không phải outcome-based feature.

Ví dụ:

```text
JFK - 2015-01-05 - 08:00
       │
       ├── flight 1
       ├── flight 2
       ├── flight 3
       └── ...
```

→ `route_congestion_count` phản ánh mật độ lịch bay ở thời điểm đó.

Code cố tình đưa `YEAR` vào group key. Nếu chỉ dùng MONTH + DAY thì ngày 15/01/2015 và 15/01/2016 sẽ bị gộp.

In [15]:
from custom_transformers import add_route_congestion_feature

congestion_demo = spark.createDataFrame(
    [
        ("JFK", 2015, 1, 5, "0800", 8),
        ("JFK", 2015, 1, 5, "0815", 8),
        ("JFK", 2015, 1, 5, "0850", 8),
        ("LAX", 2015, 1, 5, "0800", 8),
    ],
    ["ORIGIN_AIRPORT", "YEAR", "MONTH", "DAY", "SCHEDULED_DEPARTURE", "scheduled_hour"]
)

add_route_congestion_feature(congestion_demo).show()


+--------------+----+-----+---+--------------+-------------------+----------------------+
|ORIGIN_AIRPORT|YEAR|MONTH|DAY|scheduled_hour|SCHEDULED_DEPARTURE|route_congestion_count|
+--------------+----+-----+---+--------------+-------------------+----------------------+
|           JFK|2015|    1|  5|             8|               0800|                     3|
|           JFK|2015|    1|  5|             8|               0815|                     3|
|           LAX|2015|    1|  5|             8|               0800|                     1|
|           JFK|2015|    1|  5|             8|               0850|                     3|
+--------------+----+-----+---+--------------+-------------------+----------------------+



# 6. SECTION 5 — 5 cấp độ feature

Đây là phần feature engineering theo entity/time hierarchy.

| Level | Entity chính | Lookback chính |
|---|---|---|
| Airline | `AIRLINE` | 1 / 3 / 7 ngày |
| Airport | `ORIGIN_AIRPORT` | 3 / 6 / 12 giờ |
| Airplane | `TAIL_NUMBER` | 1 / 3 / 7 ngày |
| Flight | `AIRLINE + FLIGHT_NUMBER` | một số feature dùng lịch sử / thông tin chuyến |
| Temporal | ngày / giờ | không cần entity lookback |

Điểm quan trọng nhất không phải chỉ là `groupBy`.

Đó là **as-of time**:

> Khi dự đoán chuyến bay hiện tại, feature lịch sử chỉ được phép nhìn thấy những chuyến đã xảy ra trước thời điểm hiện tại.

## 6.1 Tạo `flight_timestamp`

`add_flight_timestamp_column()` ghép:

```text
YEAR + MONTH + DAY + SCHEDULED_DEPARTURE
```

thành timestamp thật.

Ngoài timestamp, code tạo:

```text
_flight_timestamp_epoch
```

để dùng với:

```python
Window.rangeBetween(...)
```

Điều này quan trọng vì lookback phải theo **thời gian thực**, không chỉ theo ngày.

In [16]:
from custom_transformers import add_flight_timestamp_column

timestamp_demo = spark.createDataFrame(
    [
        (2015, 1, 5, "0800"),
        (2015, 1, 5, "1800"),
        (2015, 1, 6, "0015"),
        (2015, 1, 6, "2400"),
    ],
    ["YEAR", "MONTH", "DAY", "SCHEDULED_DEPARTURE"]
)

add_flight_timestamp_column(timestamp_demo).show(truncate=False)


+----+-----+---+-------------------+-------------------+-----------------------+
|YEAR|MONTH|DAY|SCHEDULED_DEPARTURE|flight_timestamp   |_flight_timestamp_epoch|
+----+-----+---+-------------------+-------------------+-----------------------+
|2015|1    |5  |0800               |2015-01-05 08:00:00|1420419600             |
|2015|1    |5  |1800               |2015-01-05 18:00:00|1420455600             |
|2015|1    |6  |0015               |2015-01-06 00:15:00|1420478100             |
|2015|1    |6  |2400               |2015-01-07 00:00:00|1420563600             |
+----+-----+---+-------------------+-------------------+-----------------------+



### Vì sao `rangeBetween(-lookback_seconds, -1)`?

Ví dụ lookback 1 ngày:

```python
Window.partitionBy("AIRLINE")       .orderBy("_flight_timestamp_epoch")       .rangeBetween(-86400, -1)
```

Phần `-1` rất quan trọng.

Nó có nghĩa:

> chỉ lấy timestamp **strictly before** timestamp hiện tại.

Nhờ vậy chuyến bay lúc 08:00 không thể nhìn thấy outcome của chuyến 20:00 cùng ngày.

Nếu chỉ group theo ngày, lỗi leakage nội bộ trong ngày rất dễ xảy ra.

## 6.2 Airline-level

Mỗi cửa sổ 1 / 3 / 7 ngày tạo một bộ feature riêng.

Ví dụ với 7 ngày:

- `airline_late_flights_count_lag7d`
- `airline_total_delay_minutes_lag7d`
- `airline_flights_count_lag7d`
- `airline_distinct_tails_late_lag7d`
- `airline_late_flight_ratio_lag7d`
- `airline_delay_per_tail_lag7d`

Ngưỡng late đã chốt trong code:

```text
ARRIVAL_DELAY > 30 phút
```

Điểm quan trọng: đây là **lịch sử của các chuyến trước đó**, không phải target của chuyến hiện tại.

In [17]:
from custom_transformers import build_airline_level_features

airline_demo = spark.createDataFrame(
    [
        ("AA", 1, 1, 2015, 1, 1, "0800", 10.0, "T1"),
        ("AA", 2, 1, 2015, 1, 1, "1000", 40.0, "T2"),
        ("AA", 3, 1, 2015, 1, 1, "1200", 5.0, "T1"),
        ("DL", 4, 1, 2015, 1, 1, "0900", 50.0, "T3"),
    ],
    [
        "AIRLINE", "FLIGHT_NUMBER", "YEAR", "MONTH", "DAY",
        "SCHEDULED_DEPARTURE", "ARRIVAL_DELAY", "TAIL_NUMBER"
    ]
)

airline_demo = add_flight_timestamp_column(airline_demo)

airline_features = build_airline_level_features(
    airline_demo,
    lookback_windows=(1,)
)

airline_features.orderBy("AIRLINE", "SCHEDULED_DEPARTURE").show(truncate=False)


+-------+-------------+----+-----+---+-------------------+-------------+-----------+---+-------------------+-----------------------+--------------------------------+---------------------------------+---------------------------+---------------------------------+-------------------------------+----------------------------+
|AIRLINE|FLIGHT_NUMBER|YEAR|MONTH|DAY|SCHEDULED_DEPARTURE|ARRIVAL_DELAY|TAIL_NUMBER|_9 |flight_timestamp   |_flight_timestamp_epoch|airline_late_flights_count_lag1d|airline_total_delay_minutes_lag1d|airline_flights_count_lag1d|airline_distinct_tails_late_lag1d|airline_late_flight_ratio_lag1d|airline_delay_per_tail_lag1d|
+-------+-------------+----+-----+---+-------------------+-------------+-----------+---+-------------------+-----------------------+--------------------------------+---------------------------------+---------------------------+---------------------------------+-------------------------------+----------------------------+
|AA     |1            |1   |201

## 6.3 Airport-level

Airport có hai loại feature:

### A. Schedule-based

- `airport_scheduled_departures_same_day`
- `airport_scheduled_arrivals_same_day`

Đây là thông tin về **lịch bay**, không phải outcome.

### B. Historical delay

Cửa sổ ngắn hơn:

- 3 giờ
- 6 giờ
- 12 giờ

Ví dụ:

- `airport_late_flights_count_lag3h`
- `airport_avg_delay_lag3h`

Lý do dùng cửa sổ ngắn: disruption ở airport như ATC/weather thường mang tính ngắn hạn hơn so với pattern cấp airline.

In [18]:
from custom_transformers import build_airport_level_features

airport_demo = airline_demo.withColumn("ORIGIN_AIRPORT", F.lit("JFK"))                            .withColumn("DESTINATION_AIRPORT", F.lit("LAX"))

airport_features = build_airport_level_features(
    airport_demo,
    lookback_windows_hours=(3,)
)

airport_features.orderBy("SCHEDULED_DEPARTURE").show(truncate=False)


+-------------------+----+-----+---+--------------+-------+-------------+-------------------+-------------+-----------+---+-------------------+-----------------------+-------------------------------------+-----------------------------------+--------------------------------+-----------------------+
|DESTINATION_AIRPORT|YEAR|MONTH|DAY|ORIGIN_AIRPORT|AIRLINE|FLIGHT_NUMBER|SCHEDULED_DEPARTURE|ARRIVAL_DELAY|TAIL_NUMBER|_9 |flight_timestamp   |_flight_timestamp_epoch|airport_scheduled_departures_same_day|airport_scheduled_arrivals_same_day|airport_late_flights_count_lag3h|airport_avg_delay_lag3h|
+-------------------+----+-----+---+--------------+-------+-------------+-------------------+-------------+-----------+---+-------------------+-----------------------+-------------------------------------+-----------------------------------+--------------------------------+-----------------------+
|LAX                |1   |2015 |1  |JFK           |AA     |1            |1                  |0800      

## 6.4 Airplane-level

Entity:

```text
TAIL_NUMBER
```

Có ba nhóm ý tưởng:

1. Số chuyến được schedule trong ngày
2. Tổng delay lịch sử 1 / 3 / 7 ngày
3. Chuyến bay liền trước bằng `lag(1)`

Ví dụ:

```text
airplane_total_delay_minutes_lag7d
airplane_delay_per_km_lag7d
airplane_prev_flight_arrival_delay
```

`lag(1)` khác với rolling aggregate.

- rolling aggregate: nhìn nhiều chuyến trước trong một khoảng thời gian
- `lag(1)`: chỉ lấy đúng **một chuyến ngay trước** theo timestamp

In [19]:
from custom_transformers import build_airplane_level_features

airplane_demo = airline_demo.withColumn("distance_feature_km", F.lit(1000.0))                            .withColumn("SCHEDULED_ARRIVAL", F.lit("1000"))

airplane_features = build_airplane_level_features(
    airplane_demo,
    lookback_windows=(1,)
)

airplane_features.orderBy("TAIL_NUMBER", "SCHEDULED_DEPARTURE").show(truncate=False)


+-----------+----+-----+---+-------+-------------+-------------------+-------------+---+-------------------+-----------------------+-------------------+-----------------+-----------------------------------+----------------------------------+---------------------------+----------------------------------+----------------------------------------------+
|TAIL_NUMBER|YEAR|MONTH|DAY|AIRLINE|FLIGHT_NUMBER|SCHEDULED_DEPARTURE|ARRIVAL_DELAY|_9 |flight_timestamp   |_flight_timestamp_epoch|distance_feature_km|SCHEDULED_ARRIVAL|airplane_flights_scheduled_same_day|airplane_total_delay_minutes_lag1d|airplane_delay_per_km_lag1d|airplane_prev_flight_arrival_delay|airplane_prev_flight_delay_vs_current_schedule|
+-----------+----+-----+---+-------+-------------+-------------------+-------------+---+-------------------+-----------------------+-------------------+-----------------+-----------------------------------+----------------------------------+---------------------------+---------------------------

## 6.5 Flight-level

Flight-level dùng thông tin gắn với chính flight number/airline.

Code hiện tại tạo:

- `flight_departure_delay_per_km`
- `flight_departure_delay_vs_airline_avg_rt_lag7d` (nếu có các airline-level columns cần thiết)
- `flight_computed_scheduled_duration_minutes`
- `flight_scheduled_minutes_per_km`

### Lưu ý rất quan trọng

`DEPARTURE_DELAY` của chính chuyến đang xét chỉ hợp lệ nếu **inference contract** của bài toán cho phép biết departure delay tại thời điểm dự đoán.

Nếu bài toán là dự đoán trước khi chuyến bay khởi hành thì feature này không được phép dùng.

Code đã tách riêng `build_flight_level_features()` để nhóm dễ bật/tắt nhóm này theo contract.

## 6.6 Temporal-level

Temporal-level không cần entity.

Code hiện tại tạo:

### `is_weekend`

ISO 8601:

```text
1 = Monday
...
6 = Saturday
7 = Sunday
```

Do đó:

```python
DAY_OF_WEEK in (6, 7)
```

→ `is_weekend = 1`.

### `time_of_day_bucket`

- 05–11 → `morning`
- 12–16 → `afternoon`
- 17–20 → `evening`
- còn lại → `night`

In [20]:
from custom_transformers import build_temporal_level_features

temporal_demo = spark.createDataFrame(
    [
        (1, 8),
        (6, 10),
        (7, 18),
        (3, 23),
    ],
    ["DAY_OF_WEEK", "scheduled_hour"]
)

build_temporal_level_features(temporal_demo).show()


+-----------+--------------+----------+------------------+
|DAY_OF_WEEK|scheduled_hour|is_weekend|time_of_day_bucket|
+-----------+--------------+----------+------------------+
|          1|             8|         0|           morning|
|          6|            10|         1|           morning|
|          7|            18|         1|           evening|
|          3|            23|         0|             night|
+-----------+--------------+----------+------------------+



# 7. `safe_divide()` — một helper nhỏ nhưng quan trọng

Nhiều feature có dạng:

$$
\frac{numerator}{denominator}
$$

Ví dụ:

```text
delay / distance
total delay / flight count
late flights / total flights
```

Code không dùng phép chia trực tiếp vì denominator có thể bằng 0.

Rule hiện tại:

| Trường hợp | Kết quả |
|---|---:|
| numerator hoặc denominator NULL | NULL |
| 0 / 0 | 0 |
| dương / 0 | 99999.9999 |
| âm / 0 | -99999.9999 |
| còn lại | numerator / denominator |

Đây là rule nghiệp vụ đã được chốt trong code; notebook không thay đổi rule này.

In [21]:
from custom_transformers import safe_divide

safe_demo = spark.createDataFrame(
    [
        (10.0, 2.0),
        (0.0, 0.0),
        (10.0, 0.0),
        (-10.0, 0.0),
        (None, 2.0),
    ],
    ["num", "den"]
)

safe_demo.withColumn("result", safe_divide(F.col("num"), F.col("den"))).show()


+-----+---+-----------+
|  num|den|     result|
+-----+---+-----------+
| 10.0|2.0|        5.0|
|  0.0|0.0|        0.0|
| 10.0|0.0| 99999.9999|
|-10.0|0.0|-99999.9999|
| NULL|2.0|       NULL|
+-----+---+-----------+



# 8. Leakage map — phần cần nhớ khi thuyết trình

## 8.1 Những thứ phải fit từ train

### OutlierIQRTruncator

```text
Q1 / Q3 / IQR
```

→ chỉ tính trên train/fold.

### TargetEncoder

```text
category mean
global mean
encoding map
```

→ chỉ tính trên train/fold.

## 8.2 Những thứ có thể tính trực tiếp

### Schedule-based

Ví dụ:

```text
route_congestion_count
airport_scheduled_departures_same_day
airplane_flights_scheduled_same_day
```

Chúng mô tả **lịch bay**, không phải outcome.

## 8.3 Những thứ historical phải point-in-time

Ví dụ:

```text
airline_total_delay_minutes_lag7d
airport_avg_delay_lag6h
airplane_total_delay_minutes_lag3d
airplane_prev_flight_arrival_delay
```

→ chỉ dùng các chuyến đã xảy ra trước timestamp hiện tại.

# 9. Vì sao `custom_transformers.py` và `build_modeling_dataset.py` phải tách nhau?

## `custom_transformers.py`

Chịu trách nhiệm:

- công thức
- Spark transformation
- pandas UDF
- Estimator / Model
- Window logic
- feature definitions
- leakage-safe behavior

## `build_modeling_dataset.py`

Chịu trách nhiệm:

- thứ tự chạy
- đọc/ghi Parquet
- checkpoint
- merge vào `base`
- giải phóng memory
- cuối cùng split theo thời gian

Nói ngắn gọn:

> **custom_transformers = HOW to create features**

> **build_modeling_dataset = WHEN / IN WHAT ORDER to create and merge them**

Điều này cũng giúp tránh việc orchestration file trở thành một file feature-engineering khổng lồ.

# 10. Full pipeline logic trong project

Theo orchestration hiện tại, flow khái niệm là:

```text
1. Build BASE
   └─ eligibility filter
   └─ label_severe_delay
   └─ flight_timestamp
   └─ checkpoint Parquet

2. Haversine
   └─ airport mapping
   └─ coordinates
   └─ pandas_udf
   └─ fallback DISTANCE

3. Scheduled hour + congestion

4. Airline-level

5. Airport-level

6. Airplane-level

7. Flight-level

8. Temporal-level

9. Time-based split
   └─ first 80% time → train
   └─ last 20% time → test
```

Mỗi nhóm feature được merge về cùng grain:

> **1 row = 1 flight**

Do đó model cuối cùng nhận **một dataset duy nhất về mặt schema/grain**, thay vì mỗi nhóm feature trở thành một dataset train/test riêng.

# 11. Demo kiểm tra schema/grain

Cell này chỉ đọc `base` nếu project của bạn đã chạy pipeline trước đó.

Nếu chưa có file Parquet, bỏ qua cell này.

In [22]:
BASE_PATH = Path("data/model-ready/base")

if BASE_PATH.exists():
    base = spark.read.parquet(str(BASE_PATH))
    print("Rows:", base.count())
    print("Columns:", len(base.columns))
    print("Sample columns:")
    print(base.columns[:30])
else:
    print("Chưa có data/model-ready/base — bỏ qua demo full dataset.")


Chưa có data/model-ready/base — bỏ qua demo full dataset.


# 12. Checklist trước khi đưa feature vào model

### A. Grain

- [ ] 1 row = 1 flight
- [ ] composite key không bị duplicate
- [ ] mọi feature merge đúng composite key

### B. Leakage

- [ ] Target không được dùng để tạo feature của chính row hiện tại
- [ ] TargetEncoder chỉ fit train/fold
- [ ] IQR bounds chỉ fit train/fold
- [ ] historical window chỉ nhìn timestamp quá khứ
- [ ] feature dùng `DEPARTURE_DELAY` phải phù hợp inference contract

### C. Units

- [ ] `DISTANCE` gốc là miles
- [ ] `route_distance_km` đổi sang km bằng `1.60934`
- [ ] delay là minutes
- [ ] timestamp dùng epoch seconds cho range window

### D. Missing values

- [ ] hiểu nguyên nhân NULL
- [ ] không tự động đổi mọi NULL thành 0
- [ ] phân biệt “không có lịch sử” với “giá trị thực sự bằng 0”

### E. Distributed execution

- [ ] tránh collect dataset lớn
- [ ] chỉ collect các mapping nhỏ
- [ ] dùng pandas_udf/Arrow cho metric vectorized
- [ ] checkpoint Parquet giữa các nhóm feature khi chạy full data

### F. Model contract

- [ ] feature có thể tồn tại tại thời điểm inference
- [ ] không đưa feature hậu nghiệm vào model chỉ vì nó có correlation cao

# 13. Câu trả lời ngắn để thuyết trình

### Q1. Tại sao Haversine dùng pandas UDF?

Vì Haversine là phép tính số học trên từng dòng nhưng có thể vectorize. `pandas_udf` dùng Arrow để truyền dữ liệu theo batch và NumPy tính vectorized, tránh Python loop từng row.

### Q2. Tại sao Outlier dùng Estimator → Model?

Vì Q1/Q3 phải học từ train/fold. `fit()` học bounds; `transform()` chỉ áp dụng bounds đã học. Nhờ vậy tránh leakage.

### Q3. Tại sao Target Encoding dễ leakage?

Vì encoding được tính từ target. Nếu tính category mean bằng cả test target, thông tin của test đã đi vào feature.

### Q4. Tại sao smoothing?

Category có ít quan sát có mean rất không ổn định. Smoothing kéo mean của category về global mean khi `n` nhỏ.

### Q5. Tại sao historical feature phải dùng timestamp?

Nếu chỉ dùng ngày, một chuyến lúc 08:00 có thể vô tình nhìn thấy outcome của chuyến lúc 20:00 cùng ngày. Timestamp + `rangeBetween(..., -1)` ngăn điều đó.

### Q6. Tại sao có fallback cho Haversine?

Một số airport code không map được sang IATA nên không có tọa độ. `DISTANCE` gốc là fallback để vẫn giữ được thông tin khoảng cách.

### Q7. Tại sao không đưa clipped feature vào training?

Theo quyết định cuối của nhóm: tree-based models vốn khá robust với outlier; clipping có thể làm mất thông tin thật. Flag giữ lại tín hiệu “bất thường” mà không thay đổi giá trị gốc.

# 14. Kết luận

`custom_transformers.py` không chỉ là tập hợp các hàm tạo feature. Nó thể hiện 4 ý chính của Task 1:

1. **Distributed computation**  
   Spark Window, aggregation và `approxQuantile`.

2. **Vectorized computation**  
   Arrow + `pandas_udf` cho Haversine.

3. **Reusable Spark ML components**  
   `Estimator → Model` cho Outlier và Target Encoding.

4. **Leakage-aware feature engineering**  
   Fit statistics trên train/fold và historical feature theo point-in-time.

Đây là cách đọc file theo tư duy:

```text
Business problem
      ↓
Feature definition
      ↓
Spark implementation
      ↓
Leakage constraint
      ↓
Distributed execution
      ↓
Output feature
      ↓
Model-ready dataset
```

**Nguồn chính của notebook:** phiên bản `custom_transformers.py` cuối cùng được cung cấp trong project; orchestration được đối chiếu với `build_modeling_dataset.py` cuối cùng.